In [ ]:
import time
import json
import random

import cst_python as cst
from cst_python.memory_storage import MemoryStorageCodelet

# Inicia a mente

In [2]:
mind = cst.Mind()

surrounding_actions_mo = mind.create_memory_object("SurroundingActions")
skill_manifest_mo = mind.create_memory_object("SkillManifest")
action_command_mo = mind.create_memory_object("ActionCommand")
action_status_mo = mind.create_memory_object("ActionStatus")

mscodelet = MemoryStorageCodelet(mind, host="127.0.0.1")
mscodelet.time_step = 50
mind.insert_codelet(mscodelet)

mind.start()

In [ ]:
while surrounding_actions_mo.get_info() == "" and skill_manifest_mo.get_info() == "":
    time.sleep(1)

True

# Prepara os comandos

In [9]:
from dataclasses import dataclass, field, asdict
from typing import Any


@dataclass
class CommandEntry:
    Skill : str
    Parameters:dict[str, Any]=field(default_factory=dict)

@dataclass
class CommandPayload:
    Id:int
    Commands:list[CommandEntry]
    

In [10]:
velocity = 3.5

def create_commands(surrounding_actions:dict) -> dict[str, list[CommandEntry]]:
    result = {}

    for action in surrounding_actions:
        commands = []
        position = action["originPosition"]
        name = action["name"]

        commands.append(CommandEntry("walk_to", 
                                    {"destination":position, "velocity":velocity}))

        if name == "Trabalhar":
            commands.append(CommandEntry("work"))

        elif name == "Beber café":
            commands.append(CommandEntry("drink_coffee"))

        elif name == "Usar":
            commands.append(CommandEntry("use_bathroom"))

        result[name] = commands

    return result

In [11]:
commands = create_commands(surrounding_actions_mo.get_info())

# Loop para selecionar ações

In [ ]:
payload = CommandPayload(1, commands["Beber café"])
action_command_mo.set_info(asdict(payload))

-1

In [44]:
last_id = 1
actions = list(commands.keys())

status = action_status_mo.get_info()
last_payload_size = 2

while True:
    status = json.loads(action_status_mo.get_info())
    while status["state"] != "completed" and status["index"] < last_payload_size-1:
        time.sleep(1)
        status = json.loads(action_status_mo.get_info())

    action = random.choice(actions)
    command = commands[action]

    last_id += 1
    last_payload_size = len(command)
    payload = CommandPayload(last_id, command)

    action_command_mo.set_info(asdict(payload))

    while status["id"] != last_id:
        status = json.loads(action_status_mo.get_info())
        time.sleep(1)



KeyboardInterrupt: 